In [ ]:
using Oceananigans

Precompiling Oceananigans
  ✓ SentinelArrays
  ✓ StatsAPI
  ✓ OffsetArrays
  ✓ SeawaterPolynomials
  ✓ ProgressBars
  ✓ LaTeXStrings
  ✓ IteratorInterfaceExtensions
  ✓ ExprTools
  ✓ LLVMLoopInfo
  ✓ AbstractFFTs
  ✓ Glob
  ✓ CEnum
  ✓ CompilerSupportLibraries_jll
  ✓ StructTypes
  ✓ ErrorfreeArithmetic
  ✓ PkgVersion
  ✓ DataValueInterfaces
  ✓ Compat
  ✓ UnsafeAtomics


In [ ]:
using TopographicHorizontalConvection: HorizontalConvectionSimulation

In [ ]:
simulation = HorizontalConvectionSimulation(Ra=1e6, h₀_frac=0, Nx= 951, Ny=1, Nz=119, b_init = -0.6, advection=true, architecture=CPU())

In [ ]:
run!(simulation, pickup=false)

In [ ]:
using NCDatasets
using CairoMakie
using Printf
using Oceananigans.Fields
using Oceananigans.AbstractOperations: volume
using NaNStatistics

In [ ]:
ds = NCDataset("/pub/ikeshwan/code/HorizontalConvection/output/turbulent_h0.6_Ra1.0e6_coldstart_buoyancy.nc");

ds2 = NCDataset("/pub/ikeshwan/code/HorizontalConvection/output/turbulent_h0.6_Ra1.0e6_coldstart_oceanostics.nc");

In [ ]:
Ra = ds2.attrib["Ra"]
ν = ds2.attrib["ν"]
κ = ds2.attrib["κ"]
b★ = ds2.attrib["b★"]
H = ds2.attrib["H"]
Lx = ds2.attrib["Lx"]

In [ ]:
b = ds["b"][4+1:end-4, :, 4+1:end-4, :];
ε_sim = ds2["ε"][4+1:end-4, 1, 4+1:end-4, :];
t = ds2["time"][:];

In [ ]:
#theoretical constraint on epsilon based on our simulation parameters
ε_constraint = (κ * b★)/H

In [ ]:
function global_volume_integral(ds, var)
    x = ds["xC"][4+1:end-4]; Nx = length(x);
    z = ds["zC"][4+1:end-4]; Nz = length(z);
    time = ds["time"][:];
    Δx = reshape(diff(ds["xF"])[4+1:end-4], Nx,1,1);
    Δz = reshape(diff(ds["zF"])[4+1:end-4], 1,1,Nz);
    ΔA = Δx; #flat in y -- 2 dimensional
    ΔV = ΔA.*Δz;
    var_array = zeros(size(time,1));
    for n in 1:size(time, 1)
        var_t = ds[var][4+1:end-4, 1, 4+1:end-4, n]
        #wet = var_t.!=0.
        #var_t[.!wet] .= NaN
        var_array[n] = nansum(
            var_t .*
            ΔV,
            dims=(1,2,3)
        )[1,1,1]
    end  
    return var_array
end

In [ ]:
ε_int = global_volume_integral(ds2, "ε")
ε_avg = ε_int / (Lx * H)

In [ ]:
ε_avg[11]

In [ ]:
fig = Figure()
ax = Axis(fig[1,1], xlabel = "time", ylabel = "Volume Average ε", title = "Volume Averaged ε vs Time")
scatter!(ax, t, ε_avg)

fig

In [ ]:
f2 = Figure()
ax2 = Axis(f2[1,1], title = "comparison of ε theoretical versus simulation")
plot!(ε_avg/ε_constraint)

f2

In [ ]:
ds_3 = NCDataset("/pub/ikeshwan/code/HorizontalConvection/output/turbulent_h0.6_Ra1.0e8_coldstart_oceanostics.nc")
ds_3b = NCDataset("/pub/ikeshwan/code/HorizontalConvection/output/turbulent_h0.6_Ra1.0e8_coldstart_buoyancy.nc")

t3 = ds_3["time"][:]

In [ ]:
#lets create an epsilon vs Rayleigh number plot for paperella's constraint

b_maxes = [1, 5, 10, 15, 20]
ε_constraints = 

In [ ]:
sqrt(1e8)